# 실습 6-2 : STUCCO (대조집합 학습)

#### **<실습 내용>**

1. 대조집합 학습 개요
- STUCCO 알고리즘 원리

2. STUCCO 기본 예제 (Process_Data)
- 데이터 전처리 (연속형 변수 범주화)
- 대조집합(cset) 도출 및 해석

3. Vibe Coding 실습 (통신사 고객 이탈 데이터)

## 분석 준비

### 주요 라이브러리 호출

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import logging
from stucco import ContrastSetLearner  # 대조집합 학습(Contrast Set Learning)을 수행하는 STUCCO 알고리즘 구현체

logging.getLogger().setLevel(logging.WARNING)  # stucco.py의 상세 진행 로그 숨기기

---

## 1) 데이터 불러오기

In [2]:
data = pd.read_csv("dataset/day6-2_data.csv")
print("데이터 크기:", data.shape)
data.head()

데이터 크기: (596, 12)


,Prod_ID,M1,M2,P1,P2,P3,C1,C2,P4,P5,P6,Y
0,1001001,15,E,A,B,D,110.192,27.049,A,A,B,0
1,1001002,20,E,A,B,D,109.940,27.215,A,A,A,0
2,1001003,15,U,A,B,B,110.214,27.444,B,B,B,4
3,1001004,25,K,B,B,C,107.260,27.300,A,A,C,0
4,1001005,20,E,A,A,D,105.026,28.283,B,B,A,0


In [3]:
# 불량 기준 설정: 제품 1개당 불량 개수(Y)가 3개 이상이면 Fault, 아니면 Pass로 그룹 라벨 생성
data["Fail"] = ["Fault" if i >= 3 else "Pass" for i in data["Y"]]
data

,Prod_ID,M1,M2,P1,P2,P3,C1,C2,P4,P5,P6,Y,Fail
0,1001001,15,E,A,B,D,110.192,27.049,A,A,B,0,Pass
1,1001002,20,E,A,B,D,109.940,27.215,A,A,A,0,Pass
2,1001003,15,U,A,B,B,110.214,27.444,B,B,B,4,Fault
3,1001004,25,K,B,B,C,107.260,27.300,A,A,C,0,Pass
4,1001005,20,E,A,A,D,105.026,28.283,B,B,A,0,Pass
...,...,...,...,...,...,...,...,...,...,...,...,...,...
591,1001592,15,E,A,C,D,115.153,31.652,A,A,C,0,Pass
592,1001593,15,E,A,B,D,114.593,33.890,A,A,B,0,Pass
593,1001594,20,U,A,A,B,105.724,33.251,B,B,A,0,Pass
594,1001595,25,E,A,A,A,109.783,32.054,A,A,D,0,Pass


In [4]:
print("클래스 분포:")
print(data["Fail"].value_counts())

클래스 분포:
Fail
Pass     471
Fault    125
Name: count, dtype: int64


**활용할 반도체 공정 데이터 소개 (6-1과 동일)**

| 변수 | 설명 |
|---|---|
| Prod_ID | 제품 고유 ID |
| M1 | 공정 설비 온도 설정값 (15/20/25/30) |
| M2 | 사용 설비 ID (E, U, K) |
| P1 ~ P6 | 제품 설계/공정 옵션 (범주형 파라미터) |
| C1, C2 | 공정 중 측정된 연속형 값 |
| Y | 제품 1개당 발생한 불량 개수 |
| Fail | Y >= 3이면 Fault, 아니면 Pass로 정의한 라벨 |

## 2) 전처리하기

> STUCCO는 입력변수로 **범주형 변수만** 허용하므로 연속형 변수(C1, C2)는 구간화하여 범주형 변수 처럼 나타내야함

In [5]:
# 1. M1
# 숫자(15/20/25/30)로 저장되어 있음
# 그러나 실제로는 '온도 설정값'을 의미하는 범주이므로 범주형으로 변환
data["M1"] = data["M1"].astype("object")

# 2. 연속형 변수 C1, C2
# qcut: 데이터 개수가 균등하도록 4구간으로 분할
data["C1_cut"] = pd.qcut(data["C1"], 4)
# cut: 의미 있는 경계값 기준으로 직접 3구간 지정
data["C2_cut"] = pd.cut(data["C2"], bins=[18, 28, 32, 39], labels=["Low", "Normal", "High"]) 

### 2-1) STUCCO 실행

1. 비교할 그룹 설정
2. Contrast Set 후보 규칙 생성
3. 유의미한 규칙 추출

In [6]:
# STUCCO에 불필요한 변수 제거
# (ID, 이미 구간화한 원본 연속형 변수, 라벨 만드는 데 쓴 Y)
contra_data = data.drop(["Prod_ID", "C1", "C2", "Y"], axis=1)
contra_data

,M1,M2,P1,P2,P3,P4,P5,P6,Fail,C1_cut,C2_cut
0,15,E,A,B,D,A,A,B,Pass,"(109.191, 112.225]",Low
1,20,E,A,B,D,A,A,A,Pass,"(109.191, 112.225]",Low
2,15,U,A,B,B,B,B,B,Fault,"(109.191, 112.225]",Low
3,25,K,B,B,C,A,A,C,Pass,"(106.365, 109.191]",Low
4,20,E,A,A,D,B,B,A,Pass,"(102.158, 106.365]",Normal
...,...,...,...,...,...,...,...,...,...,...,...
591,15,E,A,C,D,A,A,C,Pass,"(112.225, 123.34]",Normal
592,15,E,A,B,D,A,A,B,Pass,"(112.225, 123.34]",High
593,20,U,A,A,B,B,B,A,Pass,"(102.158, 106.365]",High
594,25,E,A,A,A,A,A,D,Pass,"(109.191, 112.225]",High


> 1. 비교할 그룹 설정 (ContrastSetLearner())

In [7]:
# ContrastSetLearner(DataFrame, group_feature)

# "Fail" 컬럼의 그룹을 기준으로 Contrast Set 탐색
# Contrast Set: 그룹 간 차이를 가장 잘 설명하는 규칙

learner = ContrastSetLearner(contra_data, group_feature="Fail")
learner

> 2. Contrast Set 후보 규칙 생성 (learn())

In [8]:
# learner.learn():
# 그룹 간 차이를 설명할 수 있는 규칙 후보(조건 조합)를 생성
# 각 규칙의 그룹별 등장 빈도를 계산

# max_length: 규칙을 구성하는 최대 조건 수
# n_matrices: 규칙이 각 그룹에서 몇 번 나타났는지 등의 빈도 정보를 저장

n_matrices = learner.learn(max_length=3)
n_matrices

4217

> 3. 유의미한 규칙 추출 (score())

In [9]:
# learner.score():
# learn( )에서 생성한 빈도 정보를 활용해 규칙별 Support, Confidence, Lift를 계산

# Support: 규칙의 출현 비율
# Confidence: 규칙이 나타났을 때 해당 그룹일 확률
# Lift: 특정 그룹과의 연관성 정도

# lift가 2.0 이상인(특정 그룹과의 연관성 정도) 규칙만 채택
contrast_rules = learner.score(min_lift=2.0)
contrast_rules

,rule,group,lift
4,"(P1=>A, P3=>A, C2_cut=>High)",Fail=>Fault,2.666667
13,"(M2=>U, P2=>A, P3=>A)",Fail=>Fault,2.650104
7,"(M2=>K, P1=>A, P6=>D)",Fail=>Fault,2.625000
15,"(M1=>25, P1=>A, P3=>C)",Fail=>Fault,2.300000
14,"(M1=>25, P1=>A, C2_cut=>Low)",Fail=>Fault,2.300000
5,"(P1=>A, P6=>D, C1_cut=>(106.365, 109.191])",Fail=>Fault,2.258065
9,"(M2=>K, P1=>A, P2=>B)",Fail=>Fault,2.250000
10,"(P1=>A, P5=>B, C1_cut=>(112.225, 123.34])",Fail=>Fault,2.222222
8,"(P1=>A, P3=>D, P6=>D)",Fail=>Fault,2.181818
16,"(M1=>25, M2=>K, P1=>A)",Fail=>Fault,2.181818


In [10]:
# group 컬럼 값이 "Fail=>Fault", "Fail=>Pass" 형태로 저장되어 있음
# 보기 쉽게 "Fault", "Pass"만 남도록 문자열 변경
contrast_rules["group"] = contrast_rules["group"].str.replace("Fail=>", "")

# learn() 단계에서 생성된 전체 조건 조합(규칙 후보) 수 출력
print("생성된 조건 조합 수:", n_matrices)

# score() 단계의 기준을 통과하여 최종 선택된 Contrast Set 규칙 수 출력
print("도출된 규칙 수:", contrast_rules.shape[0])

생성된 조건 조합 수: 4217
도출된 규칙 수: 17


### 2-2) 결과 해석

In [11]:
# 도출된 규칙 확인
# 실제로는 Pass 규칙도 있었는데 score 기준에 따라 필터링 된 것임

contrast_rules.head()

,rule,group,lift
4,"(P1=>A, P3=>A, C2_cut=>High)",Fault,2.666667
13,"(M2=>U, P2=>A, P3=>A)",Fault,2.650104
7,"(M2=>K, P1=>A, P6=>D)",Fault,2.625000
15,"(M1=>25, P1=>A, P3=>C)",Fault,2.300000
14,"(M1=>25, P1=>A, C2_cut=>Low)",Fault,2.300000


In [12]:
# Fault 그룹을 설명하는 규칙만 추출
fault_rules = contrast_rules.loc[contrast_rules["group"] == "Fault"]

# Lift 기준 상위 5개 규칙 확인
fault_rules.sort_values(by="lift", ascending=False).head()

,rule,group,lift
4,"(P1=>A, P3=>A, C2_cut=>High)",Fault,2.666667
13,"(M2=>U, P2=>A, P3=>A)",Fault,2.650104
7,"(M2=>K, P1=>A, P6=>D)",Fault,2.625000
15,"(M1=>25, P1=>A, P3=>C)",Fault,2.300000
14,"(M1=>25, P1=>A, C2_cut=>Low)",Fault,2.300000


In [13]:
# Lift가 가장 높은 Contrast Set 규칙
# (P1=A) AND (P3=A) AND (C2_cut=High)를 만족하는 데이터 선택

cset_mask = (data["P1"] == "A") & (data["P3"] == "A") & (data["C2_cut"] == "High")

# 규칙을 만족하는 데이터의 불량(Fail) 분포 확인
print("cset 해당 데이터 불량 분포:")
print(data.loc[cset_mask, "Fail"].value_counts())
print()

# 규칙을 만족하지 않는 데이터의 불량(Fail) 분포 확인
print("cset 미해당 데이터 불량 분포:")
print(data.loc[~cset_mask, "Fail"].value_counts())

cset 해당 데이터 불량 분포:
Fail
Fault    12
Pass      6
Name: count, dtype: int64

cset 미해당 데이터 불량 분포:
Fail
Pass     465
Fault    113
Name: count, dtype: int64


---

## 3) Vibe Coding 실습 (통신사 고객 이탈 데이터)

STUCCO를 **통신사 고객 이탈(Customer Churn) 데이터셋**에 직접 적용해 봅니다.

- `dataset/day6_vibecoding.csv` 파일 사용하기
- 타겟 변수: `Churn` (No: 유지, Yes: 이탈)

**[과제]** 본 데이터셋에서의 연속형 범수인 'tenure', 'MonthlyCharges', 'TotalCharges'를 `pd.qcut` 등으로 구간화하고 `ContrastSetLearner`로 Churn(Yes/No) 그룹 간 대조집합을 도출하는 코드를 AI와 상의해서 작성하세요.

**[과제]** 도출된 규칙 중 이탈(Yes) 그룹에서 lift가 높은 상위 규칙들을 확인하고 어떤 조건 조합이 이탈과 강하게 연관되는지 AI와 함께 해석해 보세요.

In [23]:
import pandas as pd
import numpy as np
from itertools import product
from sklearn.ensemble import RandomForestClassifier
from stucco import ContrastSetLearner

# ============================================================
# 1) X에서 연속형 3개 구간화 (pd.qcut)
# ============================================================
X_stucco = X.copy()

X_stucco["tenure_cut"] = pd.qcut(
    X["tenure"], q=4, duplicates="drop"
)
X_stucco["MonthlyCharges_cut"] = pd.qcut(
    X["MonthlyCharges"], q=4, duplicates="drop"
)
X_stucco["TotalCharges_cut"] = pd.qcut(
    X["TotalCharges"], q=4, duplicates="drop"
)

# 원본 연속형 제거
X_stucco = X_stucco.drop(columns=["tenure", "MonthlyCharges", "TotalCharges"])

print("=== 구간화 결과 ===")
for col in ["tenure_cut", "MonthlyCharges_cut", "TotalCharges_cut"]:
    print(f"\n[{col}]")
    print(X_stucco[col].value_counts().sort_index())

# ============================================================
# 2) Random Forest로 영향력 상위 10개 변수 선정
# ============================================================
y = churn.map({"No": 0, "Yes": 1})

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"   # 이탈(Yes) 비율이 낮을 때 유리
)
rf.fit(X, y)

importance = pd.Series(rf.feature_importances_, index=X.columns)
top10_features = importance.sort_values(ascending=False).head(10)

print("\n=== Random Forest 상위 10개 변수 ===")
display(top10_features.to_frame("importance"))

top10_list = top10_features.index.tolist()

# 연속형 → qcut 컬럼명으로 매핑
def map_to_stucco_col(col_name):
    mapping = {
        "tenure": "tenure_cut",
        "MonthlyCharges": "MonthlyCharges_cut",
        "TotalCharges": "TotalCharges_cut",
    }
    return mapping.get(col_name, col_name)

top10_stucco_cols = [map_to_stucco_col(c) for c in top10_list]
print("\nSTUCCO에 사용할 컬럼:", top10_stucco_cols)

# ============================================================
# 3) 상위 10개 변수로 ContrastSetLearner + 파라미터 탐색
# ============================================================

# STUCCO 입력 데이터 구성
contra_data = X_stucco[top10_stucco_cols].copy()
contra_data["Churn"] = churn.values

# 숫자(0/1 더미 등) → 문자열 변환 (STUCCO가 범주형으로 인식하도록)
for col in top10_stucco_cols:
    contra_data[col] = contra_data[col].astype(str)

# 실행 전 확인
print("\n=== contra_data 확인 ===")
print("shape:", contra_data.shape)
print("Churn 분포:\n", contra_data["Churn"].value_counts())
print("결측치:", contra_data.isnull().sum().sum())

# 파라미터 그리드 탐색
max_lengths = [2, 3, 4]
min_lifts = [1.2, 1.5, 2.0, 2.5, 3.0]

search_results = []
all_rules_list = []

for max_length, min_lift in product(max_lengths, min_lifts):
    learner = ContrastSetLearner(contra_data.copy(), group_feature="Churn")  # ← copy()
    n_matrices = learner.learn(max_length=max_length)
    rules = learner.score(min_lift=min_lift)

    if len(rules) > 0:
        rules = rules.copy()
        rules["group"] = rules["group"].str.replace("Churn=>", "", regex=False)
        yes_rules = rules[rules["group"] == "Yes"]
        no_rules = rules[rules["group"] == "No"]
        max_lift_yes = yes_rules["lift"].max() if len(yes_rules) > 0 else 0
        max_lift_no = no_rules["lift"].max() if len(no_rules) > 0 else 0
    else:
        yes_rules = pd.DataFrame()
        no_rules = pd.DataFrame()
        max_lift_yes = 0
        max_lift_no = 0

    search_results.append({
        "max_length": max_length,
        "min_lift": min_lift,
        "n_matrices": n_matrices,
        "n_rules_total": len(rules),
        "n_rules_yes": len(yes_rules),
        "n_rules_no": len(no_rules),
        "max_lift_yes": max_lift_yes,
        "max_lift_no": max_lift_no,
    })

    if len(rules) > 0:
        rules["max_length"] = max_length
        rules["min_lift"] = min_lift
        all_rules_list.append(rules)

results_df = pd.DataFrame(search_results)
results_df = results_df.sort_values(
    by=["max_lift_yes", "n_rules_yes", "n_rules_total"],
    ascending=False,
)

print("\n=== 파라미터 탐색 결과 ===")
display(results_df)

# Yes 규칙이 나온 조합 중 최적 선택
valid_results = results_df[results_df["n_rules_yes"] > 0]

if len(valid_results) == 0:
    print("Yes 규칙이 없습니다. min_lift를 낮추거나 max_length를 늘려보세요.")
else:
    best = valid_results.iloc[0]
    best_max_length = int(best["max_length"])
    best_min_lift = float(best["min_lift"])

    print(f"\n최적 파라미터: max_length={best_max_length}, min_lift={best_min_lift}")

    best_learner = ContrastSetLearner(contra_data.copy(), group_feature="Churn")  
    best_n_matrices = best_learner.learn(max_length=best_max_length)
    best_rules = best_learner.score(min_lift=best_min_lift)
    best_rules["group"] = best_rules["group"].str.replace("Churn=>", "", regex=False)

    yes_rules = best_rules[best_rules["group"] == "Yes"].sort_values(
        "lift", ascending=False
    )

    print(f"생성된 조건 조합 수: {best_n_matrices}")
    print(f"도출된 규칙 수: {len(best_rules)}")

    print("\n=== 이탈(Yes) Lift 상위 규칙 ===")
    display(yes_rules.head(10))

=== 구간화 결과 ===

[tenure_cut]
tenure_cut
(-1.329, -0.959]    1817
(-0.959, -0.137]    1733
(-0.137, 0.931]     1780
(0.931, 1.629]      1713
Name: count, dtype: int64

[MonthlyCharges_cut]
MonthlyCharges_cut
(-1.5659999999999998, -0.867]    1761
(-0.867, 0.185]                  1764
(0.185, 0.834]                   1761
(0.834, 1.814]                   1757
Name: count, dtype: int64

[TotalCharges_cut]
TotalCharges_cut
(-1.0, -0.83]     1761
(-0.83, -0.39]    1766
(-0.39, 0.664]    1755
(0.664, 2.827]    1761
Name: count, dtype: int64

=== Random Forest 상위 10개 변수 ===


,importance
TotalCharges,0.178220
tenure,0.168957
MonthlyCharges,0.149344
Contract_Two year,0.054187
InternetService_Fiber optic,0.042717
PaymentMethod_Electronic check,0.034039
Contract_One year,0.029059
OnlineSecurity_Yes,0.027711
gender,0.025196
PaperlessBilling,0.024055



STUCCO에 사용할 컬럼: ['TotalCharges_cut', 'tenure_cut', 'MonthlyCharges_cut', 'Contract_Two year', 'InternetService_Fiber optic', 'PaymentMethod_Electronic check', 'Contract_One year', 'OnlineSecurity_Yes', 'gender', 'PaperlessBilling']

=== contra_data 확인 ===
shape: (7043, 11)
Churn 분포:
 Churn
No     5174
Yes    1869
Name: count, dtype: int64
결측치: 0

=== 파라미터 탐색 결과 ===


,max_length,min_lift,n_matrices,n_rules_total,n_rules_yes,n_rules_no,max_lift_yes,max_lift_no
10,4,1.2,10174,2047,1026,1021,6.118217,2.187623
11,4,1.5,10174,1153,719,434,6.118217,2.187623
12,4,2.0,10174,375,367,8,6.118217,2.187623
13,4,2.5,10174,217,217,0,6.118217,0.000000
14,4,3.0,10174,110,110,0,6.118217,0.000000
5,3,1.2,2267,585,263,322,5.452207,2.024630
6,3,1.5,2267,264,175,89,5.452207,2.024630
7,3,2.0,2267,62,61,1,5.452207,2.024630
8,3,2.5,2267,38,38,0,5.452207,0.000000
9,3,3.0,2267,15,15,0,5.452207,0.000000



최적 파라미터: max_length=4, min_lift=1.2
생성된 조건 조합 수: 10174
도출된 규칙 수: 2047

=== 이탈(Yes) Lift 상위 규칙 ===


,rule,group,lift
1638,"(tenure_cut=>(-1.329, -0.959], MonthlyCharges_...",Yes,6.118217
1788,"(TotalCharges_cut=>(-1.0, -0.83], MonthlyCharg...",Yes,6.105473
1122,"(tenure_cut=>(-0.959, -0.137], MonthlyCharges_...",Yes,5.851575
1823,"(TotalCharges_cut=>(-1.0, -0.83], MonthlyCharg...",Yes,5.664777
1520,"(TotalCharges_cut=>(-0.83, -0.39], MonthlyChar...",Yes,5.534884
1517,"(TotalCharges_cut=>(-0.83, -0.39], MonthlyChar...",Yes,5.452207
1525,"(TotalCharges_cut=>(-0.83, -0.39], MonthlyChar...",Yes,5.355014
1524,"(TotalCharges_cut=>(-0.83, -0.39], MonthlyChar...",Yes,5.280722
434,"(MonthlyCharges_cut=>(-0.867, 0.185], Contract...",Yes,5.086067
459,"(MonthlyCharges_cut=>(-0.867, 0.185], Contract...",Yes,4.909442


**[과제]** 가장 중요한 규칙에 해당하는 고객들의 실제 이탈률을 비교해서 해당 규칙이 실제로 유의미한지 검증해 보세요.

In [24]:
def rule_to_mask(df, rule_tuple, sep="=>"):
    """STUCCO rule tuple → boolean mask"""
    mask = pd.Series(True, index=df.index)
    for condition in rule_tuple:
        col, val = condition.split(sep, 1)
        mask &= (df[col].astype(str) == val)
    return mask

# 최상위 Yes 규칙
top_rule = yes_rules.iloc[0]
rule_tuple = top_rule["rule"]

print("=" * 50)
print("최상위 규칙:", rule_tuple)
print(f"Lift: {top_rule['lift']:.2f}")
print("=" * 50)

# contra_data는 검증용으로 copy 사용 (원본 보존)
verify_data = contra_data.copy()
cset_mask = rule_to_mask(verify_data, rule_tuple)

# 전체 이탈률
overall_rate = (verify_data["Churn"] == "Yes").mean()
print(f"\n전체 이탈률: {overall_rate:.2%}")

# 규칙 해당 고객
print("\n[규칙 해당 고객]")
print(verify_data.loc[cset_mask, "Churn"].value_counts())
in_rate = (verify_data.loc[cset_mask, "Churn"] == "Yes").mean()
print(f"이탈률: {in_rate:.2%}  (n={cset_mask.sum()})")

# 규칙 미해당 고객
print("\n[규칙 미해당 고객]")
print(verify_data.loc[~cset_mask, "Churn"].value_counts())
out_rate = (verify_data.loc[~cset_mask, "Churn"] == "Yes").mean()
print(f"이탈률: {out_rate:.2%}  (n={(~cset_mask).sum()})")

# 차이
print(f"\n이탈률 차이: {(in_rate - out_rate):.2%}p")
print(f"Lift 검증: {in_rate / overall_rate:.2f} (STUCCO Lift: {top_rule['lift']:.2f})")

최상위 규칙: ('tenure_cut=>(-1.329, -0.959]', 'MonthlyCharges_cut=>(0.834, 1.814]', 'Contract_Two year=>False', 'PaperlessBilling=>1')
Lift: 6.12

전체 이탈률: 26.54%

[규칙 해당 고객]
Churn
Yes    99
No     30
Name: count, dtype: int64
이탈률: 76.74%  (n=129)

[규칙 미해당 고객]
Churn
No     5144
Yes    1770
Name: count, dtype: int64
이탈률: 25.60%  (n=6914)

이탈률 차이: 51.14%p
Lift 검증: 2.89 (STUCCO Lift: 6.12)
